# ML-Driven DCF Valuation Engine -- Part 2: ML Training & DCF Valuation

```
FMP Financial Data          (Part 1)
        |
Merge Financial Statements  (Part 1)
        |
Feature Engineering         (Part 1)
        |
Train ML Models             <-- YOU ARE HERE
        |
Forecast & DCF Valuation    <-- YOU ARE HERE
        |
Compare vs Market Price     (Part 3)
        |
Buy / Hold / Sell Signal    (Part 3)
```

---
**Prerequisites:** Run Part 1 first, or run the setup cell below to load and prepare the data.

### Setup -- Re-run Data Loading & Feature Engineering from Part 1

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

# -- Load CSVs --
income     = pd.read_csv('fmp_dataset/csv/income-statement.csv')
balance    = pd.read_csv('fmp_dataset/csv/balance-sheet-statement.csv')
cashflow   = pd.read_csv('fmp_dataset/csv/cash-flow-statement.csv')
enterprise = pd.read_csv('fmp_dataset/csv/enterprise-values.csv')
quote      = pd.read_csv('fmp_dataset/csv/quote.csv')
dcf_fmp    = pd.read_csv('fmp_dataset/csv/discounted-cash-flow.csv')

# -- Select columns & merge --
inc_cols = ['date','symbol','revenue','costOfRevenue','grossProfit',
    'researchAndDevelopmentExpenses','sellingGeneralAndAdministrativeExpenses',
    'operatingExpenses','operatingIncome','ebit','ebitda',
    'depreciationAndAmortization','interestExpense',
    'incomeBeforeTax','incomeTaxExpense','netIncome',
    'eps','epsDiluted','weightedAverageShsOut','weightedAverageShsOutDil']
bs_cols = ['date','symbol','cashAndCashEquivalents','shortTermInvestments',
    'netReceivables','inventory','totalCurrentAssets',
    'propertyPlantEquipmentNet','goodwillAndIntangibleAssets',
    'totalAssets','accountPayables','totalCurrentLiabilities',
    'longTermDebt','totalLiabilities','totalStockholdersEquity','totalDebt','netDebt']
cf_cols = ['date','symbol','stockBasedCompensation','changeInWorkingCapital',
    'operatingCashFlow','capitalExpenditure','freeCashFlow',
    'acquisitionsNet','netCashProvidedByOperatingActivities',
    'commonStockRepurchased','commonDividendsPaid']
ev_cols = ['date','symbol','stockPrice','numberOfShares','marketCapitalization','enterpriseValue']

merged = income[inc_cols].merge(balance[bs_cols], on=['symbol','date'], how='inner')
merged = merged.merge(cashflow[cf_cols], on=['symbol','date'], how='inner')
merged = merged.merge(enterprise[ev_cols], on=['symbol','date'], how='inner')
merged['date'] = pd.to_datetime(merged['date'])
merged = merged.sort_values(['symbol','date']).reset_index(drop=True)

# -- Feature engineering --
def engineer_features(group):
    g = group.copy()
    rev = g['revenue']
    g['rev_growth'] = rev.pct_change()
    g['gross_margin']  = g['grossProfit'] / rev
    g['ebit_margin']   = g['ebit'] / rev
    g['eff_tax_rate'] = (g['incomeTaxExpense'] / g['incomeBeforeTax']).clip(0, 0.40)
    g['da_pct_rev']    = g['depreciationAndAmortization'] / rev
    g['capex_pct_rev'] = g['capitalExpenditure'].abs() / rev
    g['nwc']           = g['totalCurrentAssets'] - g['totalCurrentLiabilities']
    g['nwc_pct_rev']   = g['nwc'] / rev
    g['delta_nwc']     = g['nwc'].diff()
    driver_cols = ['rev_growth','gross_margin','ebit_margin',
                   'eff_tax_rate','da_pct_rev','capex_pct_rev','nwc_pct_rev']
    for col in driver_cols:
        g[f'{col}_lag1'] = g[col].shift(1)
    return g

merged = merged.groupby('symbol', group_keys=False).apply(engineer_features)
merged = merged.replace([np.inf, -np.inf], np.nan)
print(f'[Setup] Merged dataset ready: {merged.shape[0]} rows x {merged.shape[1]} columns')

---
## Step 4 -- Train ML Models
We train one **Random Forest Regressor** per business driver. The model learns cross-sectional patterns across all 14 companies (e.g., "companies with high gross margins tend to maintain them").

**Why Random Forest?**
- Handles non-linear relationships between financial ratios.
- Robust to outliers (common in financial data).
- Built-in feature importance for interpretability.
- No feature scaling required.

In [ ]:
# -- Define features (lagged drivers) and targets (current drivers) --
FEATURES = [
    'rev_growth_lag1', 'gross_margin_lag1', 'ebit_margin_lag1',
    'eff_tax_rate_lag1', 'da_pct_rev_lag1', 'capex_pct_rev_lag1', 'nwc_pct_rev_lag1'
]

TARGETS = [
    'rev_growth', 'gross_margin', 'ebit_margin',
    'eff_tax_rate', 'da_pct_rev', 'capex_pct_rev', 'nwc_pct_rev'
]

# -- Training data: rows where all features AND targets exist --
train_df = merged.dropna(subset=FEATURES + TARGETS).copy()
print(f'Training samples: {len(train_df)} (from {train_df["symbol"].nunique()} companies)')

# -- Train one model per target driver --
models = {}
model_scores = {}

for target in TARGETS:
    X = train_df[FEATURES]
    y = train_df[target]

    rf = RandomForestRegressor(
        n_estimators=200,
        max_depth=4,
        min_samples_leaf=3,
        random_state=42
    )
    rf.fit(X, y)
    models[target] = rf

    # Cross-validated R2 score for reporting
    cv_scores = cross_val_score(rf, X, y, cv=min(3, len(train_df)), scoring='r2')
    model_scores[target] = cv_scores.mean()

# -- Display model performance --
score_df = pd.DataFrame({
    'Business Driver': TARGETS,
    'Cross-Val R2': [f'{model_scores[t]:.3f}' for t in TARGETS]
})
print('\n-- Model Performance (Cross-Validated R2) --')
print(score_df.to_string(index=False))

---
## Steps 5 & 6 -- Forecast Business Drivers -> Forecast Financial Statements
For each company, we take the most recent year's drivers as input, then **iteratively forecast 5 years forward**. Each year's predictions become the next year's lagged features.

From the forecasted drivers, we reconstruct the line items needed for Free Cash Flow to Firm (FCFF):

```
FCFF = NOPAT + D&A - Capex - dNWC
     = EBIT x (1 - Tax Rate) + D&A - Capex - dNWC
```

In [ ]:
# -- DCF Assumptions --
WACC                = 0.09    # 9% weighted average cost of capital
TERMINAL_GROWTH     = 0.025   # 2.5% perpetual growth (long-run GDP)
FORECAST_YEARS      = 5

# -- Get the latest fiscal year for each company --
latest = merged.sort_values('date').groupby('symbol').tail(1).copy()
latest = latest.dropna(subset=FEATURES)  # need valid features to start forecasting

print(f'Running DCF for {len(latest)} companies...\n')

all_results = []
all_projections = {}  # store detailed projections per company

for _, row in latest.iterrows():
    sym = row['symbol']

    # -- Starting state --
    curr_rev = row['revenue']
    curr_nwc = row['nwc']

    # Build the feature vector from the latest year's actual drivers
    feat_dict = {f: row[f] for f in FEATURES}
    feat_input = pd.DataFrame([feat_dict])

    yearly_detail = []
    fcff_list = []

    for yr in range(1, FORECAST_YEARS + 1):
        # -- Step 5: Predict next year's drivers --
        preds = {t: models[t].predict(feat_input)[0] for t in TARGETS}

        # -- Step 6: Reconstruct financial statement line items --
        proj_rev   = curr_rev * (1 + preds['rev_growth'])
        proj_gross = proj_rev * preds['gross_margin']
        proj_ebit  = proj_rev * preds['ebit_margin']
        proj_tax   = proj_ebit * preds['eff_tax_rate']
        proj_nopat = proj_ebit - proj_tax
        proj_da    = proj_rev * preds['da_pct_rev']
        proj_capex = proj_rev * preds['capex_pct_rev']
        proj_nwc   = proj_rev * preds['nwc_pct_rev']
        d_nwc      = proj_nwc - curr_nwc

        # -- FCFF = NOPAT + D&A - Capex - dNWC --
        fcff = proj_nopat + proj_da - proj_capex - d_nwc
        fcff_list.append(fcff)

        yearly_detail.append({
            'Year': yr,
            'Revenue': proj_rev,
            'Rev Growth': preds['rev_growth'],
            'EBIT Margin': preds['ebit_margin'],
            'NOPAT': proj_nopat,
            'DA': proj_da,
            'Capex': proj_capex,
            'Delta NWC': d_nwc,
            'FCFF': fcff
        })

        # -- Roll forward: this year's predictions become next year's lags --
        feat_dict = {f'{t}_lag1': preds[t] for t in TARGETS}
        feat_input = pd.DataFrame([feat_dict])
        curr_rev = proj_rev
        curr_nwc = proj_nwc

    all_projections[sym] = pd.DataFrame(yearly_detail)

    # -- Step 7: DCF Valuation --
    discount_factors = [(1 + WACC) ** yr for yr in range(1, FORECAST_YEARS + 1)]
    pv_fcff = sum(f / d for f, d in zip(fcff_list, discount_factors))

    terminal_value    = (fcff_list[-1] * (1 + TERMINAL_GROWTH)) / (WACC - TERMINAL_GROWTH)
    pv_terminal_value = terminal_value / discount_factors[-1]

    enterprise_value  = pv_fcff + pv_terminal_value

    # -- Step 8: Intrinsic Value per Share --
    cash       = row['cashAndCashEquivalents']
    debt       = row['totalDebt']
    shares     = row['weightedAverageShsOutDil']

    equity_value = enterprise_value + cash - debt
    intrinsic_per_share = equity_value / shares if shares > 0 else np.nan

    all_results.append({
        'Symbol': sym,
        'Last FY Revenue': row['revenue'],
        'PV of FCFFs': pv_fcff,
        'PV of Terminal Value': pv_terminal_value,
        'Enterprise Value': enterprise_value,
        '(+) Cash': cash,
        '(-) Debt': debt,
        'Equity Value': equity_value,
        'Shares Outstanding': shares,
        'ML Intrinsic Value': intrinsic_per_share
    })

valuation = pd.DataFrame(all_results)
print('[DONE] Forecasting & DCF complete for all companies.')

### Projected Financial Statements (Example: AAPL)
Show the year-by-year ML-forecasted financials for Apple.

In [ ]:
# -- Display the 5-year projection for one company --
example_sym = 'AAPL'
proj = all_projections[example_sym].copy()

# Format large numbers in billions for readability
money_cols = ['Revenue', 'NOPAT', 'DA', 'Capex', 'Delta NWC', 'FCFF']
for col in money_cols:
    proj[col] = proj[col].apply(lambda x: f'${x/1e9:.1f}B')
proj['Rev Growth']  = proj['Rev Growth'].apply(lambda x: f'{x:.1%}')
proj['EBIT Margin'] = proj['EBIT Margin'].apply(lambda x: f'{x:.1%}')

print(f'\n-- 5-Year Projected Financials: {example_sym} --')
proj